# ಪಾಠ 05 - ಏಜೆಂಟಿಕ್ RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಏನು Agentic RAG?

ಪರಂಪರাগত RAG ಒಂದು ನಿಗದಿತ ಪ್ರಕ್ರಿಯೆಯನ್ನು ಅನುಸರಿಸುತ್ತದೆ: ದಾಖಲೆಗಳನ್ನು ಪಡೆಯುವುದು, ನಂತರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ರಚಿಸುವುದು. **Agentic RAG** ಏಜೆಂಟ್‌ಗೆ ಸ್ವತಂತ್ರತೆಯನ್ನು ನೀಡುತ್ತದೆ **ಯಾವಾಗ** ಮತ್ತು **ಹೆಗೆ** ಮಾಹಿತಿಯನ್ನು ಪಡೆದುಕೊಳ್ಳಬೇಕೆಂದು ತೀರ್ಮಾನಿಸಲು.

Agentic RAG ಬಳಸಿ, ಏಜೆಂಟ್ ಕೆಳಗಿನ ಕಾರ್ಯಗಳನ್ನು ಮಾಡಬಹುದು:
- **ತೀರ್ಮಾನಿಸುವುದು** ಪ್ರತಿಕ್ರಿಯಿಸುವ ಮುನ್ನ ಮಾಹಿತಿಯನ್ನು ಪಡೆಯಬೇಕೇ ಇಲ್ಲವೇ ಎನ್ನುವುದನ್ನು
- **ಆಯ್ಕೆಮಾಡುವುದು** ಯಾವ ಡೇಟಾ ಮೂಲ ಅಥವಾ ಸಾಧನವನ್ನು ಪ್ರಶ್ನಿಸುವುದನ್ನು
- **ಮೌಲ್ಯಮಾಪನ** ಪಡೆದ ಫಲಿತಾಂಶಗಳನ್ನು ಮತ್ತು ಮೊದಲ ಪ್ರಯತ್ನ ತೃಪ್ತಿಕರವಾಗದಿದ್ದರೆ ಮುಂದುವರಿದ ಓಟಗಳನ್ನು ನಡೆಸುವುದು
- **ಸಂಯೋಜಿಸುವುದು** ಅನೇಕ ಪಡೆಯುವ ಹಂತಗಳಿಂದ ಮಾಹಿತಿಯನ್ನು ಒಂದು ಸुसಂಗತ ಉತ್ತರವಾಗಿ

ಇದು ಏಜೆಂಟ್ ಅನ್ನು ಸ್ಥಿರ "ಪಡೆಯು-ಮತ್ತು-ರಚಿಸು" ಪ್ರಕ್ರಿಯೆಗೆ ಹೋಲಿಸಿದಾಗ ಹೆಚ್ಚು ಸಾಕಷ್ಟು ಮತ್ತು ನಿಖರವಾಗಿಸುತ್ತದೆ.


## ಹುಡುಕಾಟ ಸಾಧನವನ್ನು ರಚಿಸುವುದು

ಏಜೆಂಟಿಕ್ RAG ನಲ್ಲಿ, ಬಾಹ್ಯ ಡೇಟಾ ಮೂಲಗಳನ್ನು ಏಜೆಂಟ್ ಅಗತ್ಯವನ್ನು ಅನುಸರಿಸಿ ಕರೆ ಮಾಡಬಹುದಾದ **ಸಾಧನಗಳಾಗಿ** ಮುಚ್ಚಲಾಗುತ್ತದೆ. ಇದರಿಂದ ಏಜೆಂಟ್ ಹುಡುಕಾಟವನ್ನು ಒಂದು ಅತಿವಶ್ಯಕ ಹಂತವಲ್ಲದೆ, ಅದು ಕೈಗೊಂಡುಬಲ್ಲ ಇನ್ನೊಂದು ಕ್ರಿಯೆಯಾಗಿಸಿದೆ ಎಂದು ಪರಿಗಣಿಸಬಹುದು.

ಕೆಳಗೆ ನಾವು ಪ್ರಯಾಣ ಜ್ಞಾನ ಭಂಡಾರವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ, ಗಮ್ಯಸ್ಥಾನ ಮಾಹಿತಿಯನ್ನು ಹುಡುಕಲು ಏಜೆಂಟ್ ಕರೆ ಮಾಡಬಹುದಾದ ಸಾಧನವಾಗಿ ಅದನ್ನು ಬಹಿರಂಗಪಡಿಸುತ್ತೇವೆ.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ಏಜೆಂಟ್ ನಿರ್ಮಾಣ

ಈಗ ನಾವು ತಯಾರಿಸುವ ಏಜೆಂಟ್‌ಗೆ **ಎಲ್ಲಾ ಬಾರಿ ಉತ್ತರಿಸುವ ಮೊದಲು ಮಾಹಿತಿ ಪಡೆದುಕೊಳ್ಳಲು** ಸೂಚಿಸಲಾಗುತ್ತದೆ. ಈ ಏಜೆಂಟ್ ತನ್ನ ಉತ್ತರಗಳನ್ನು ತನ್ನ ಸ್ವಂತ ತರಬೇತಿ ಡೇಟಾದ ಮೇಲೆ ಅವಲಂಬಿಸದೇ ಜ್ಞಾನಾಧಾರದಲ್ಲಿ ನೆಲಗೊಳಿಸಲು `search_travel_knowledge` ಉಪಕರಣವನ್ನು ಬಳಸುತ್ತದೆ.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ಪರಾವರ್ತಕ ತೆಗೆಯುವುದು — ಮೇಕರ್-ಚೆಕ್‍ರ್ ಮಾದರಿ

ಏಜೆಂಟಿಕ್ RAG ನ ಪ್ರಮುಖ ಲಾಭವೆಂದರೆ **ಪರಾವರ್ತಕ ತೆಗೆಯುವಿಕೆ**. ಏಜೆಂಟ್ ತನ್ನ ಪ್ರಾರಂಭಿಕ ಕಂಡುಹಿಡಿವಿಕೆಯನ್ನು ಪರಿಶೀಲಿಸಲು, ಸುಧಾರಿಸಲು ಅಥವಾ ವಿಸ್ತರಿಸಲು ಅನೇಕ ಸಿರcles ಗಳನ್ನು ಶೋಧನೆ ನಡೆಸಬಹುದು — ಇದು "ಮೇಕರ್-ಚೆಕ್‍ರ್" ಕೆಲಸದ ಹಾದಿಗೆ ಸಮಾನವಾಗಿದೆ:

1. **ಮೇಕರ್ ಹಂತ**: ಏಜೆಂಟ್ ಪ್ರಾಥಮಿಕ ಮಾಹಿತಿಯನ್ನು ತೆಗೆದು ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ತಯಾರಿಸುತ್ತದೆ.
2. **ಚೆಕ್‍ರ್ ಹಂತ**: ಏಜೆಂಟ್ ವಿವರಗಳನ್ನು ಪರಿಶೀಲಿಸಲು ಅಥವಾ ಖಾಲಿ ಸ್ಥಳಗಳನ್ನು ಭರಿಸಲು ಹೆಚ್ಚುವರಿ ತೆಗೆಯುವಿಕೆಗಳನ್ನು ಮಾಡುತ್ತದೆ.

ಕೆಳಗೆ, ಏಜೆಂಟ್ನಿಗೆ ಬಹುಮಾನಸ್ಥಾನಗಳನ್ನು ಹೋಲಿಕೆಯು ಅಗತ್ಯವಿರುವ ಪ್ರಶ್ನೆ ಕೇಳಲಾಗಿದೆ, ಇದು ಅದನ್ನು ಅನೇಕ ಬಾರಿ ಶೋಧಿಸಲು ಪ್ರೇರೇಪಿಸುತ್ತದೆ.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework ಉಪಯೋಗಿಸಿ **Agentic RAG** ವ್ಯವಸ್ಥೆಯನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಬೇಕೆನ್ನುವುದನ್ನು ಕಲಿತಿರಿ:

- **Agentic RAG** ಎಜೆಂಟ್‌ಗಳಿಗೆ ಸ್ವತಃ ಸರ್ವ ತೋರುವ ಸಮಯವನ್ನು ತೀರ್ಮಾನಿಸುವುದನ್ನು ಅನುಮತಿಸುತ್ತದೆ, ಇದರಿಂದ ತೊರುವಿಕೆ ಸ್ಥಿರವಲ್ಲದೆ ಚಲನೆಯಾಗಿರುತ್ತದೆ.
- **ಸಾಧನಗಳು ಡೇಟಾ ಮೂಲಗಳಾಗಿ**: ಬಾಹ್ಯ ಜ್ಞಾನಾಧಾರಗಳು (Azure AI Search ಇತ್ಯಾದಿ) ವ್ಯವಸ್ಥೆಯ ಸಾಧನಗಳಾಗಿ ಮಡತೆಗೊಳ್ಳುತ್ತವೆ, ಇದನ್ನು ಎಜೆಂಟ್ ಕರೆಮಾಡಬಹುದು.
- **ಪುನರಾವರ್ತಿತ ತೊರುವಿಕೆ**: ನಿರ್ಮಾಪಕ-ಪರಿಶೋಧಕ ಮಾದರಿ ಎಜೆಂಟ್‌ಗೆ ಹಲವಾರು ತೊರುವಿಕೆ ಸುತ್ತುಗಳನ್ನು ನಡೆಸಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ — ಹುಡುಕಾಟ, ಪರಿಶೀಲನೆ, ಮತ್ತು ಸುಧಾರಣೆ — ಕೊನೆಗಿನ ಉತ್ತರವನ್ನು ನೀಡುವ ಮೊದಲು.

ಉತ್ಪಾದನೆಯಲ್ಲಿ, ನೀವು ಸ್ಮೃತಿ ಆಧಾರದ `TRAVEL_KNOWLEDGE_BASE` ಅನ್ನು ಬದಲಿಸಿ ದೊಡ್ಡ ಮಟ್ಟದ ಪ್ರಯಾಣ ದಾಖಲಾತಿ ತೊರುವಿಕೆಗೆ ನಿಜವಾದ Azure AI Search ಸೂಚ್ಯಂಕವನ್ನು ಬಳಸುತ್ತೀರಿ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
